## 실습 7 — 열설비 데이터 해석

사용 파일
- `03-01_유압·열설비_신호_계통태그목록.csv`
- `03-01_유압·열설비_신호_가열로온도.csv`


In [27]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)


In [28]:
tag_path = os.path.join("..", "Data", "03-01_유압·열설비_신호_계통태그목록.csv")
temp_path = os.path.join("..", "Data", "03-01_유압·열설비_신호_가열로온도.csv")

tags = pd.read_csv(tag_path, encoding="utf-8")
df = pd.read_csv(temp_path, encoding="utf-8")

print(df.head().to_markdown(index=False))


| date       |   FUR01_Z1_TEMP_L |   FUR01_Z1_TEMP_R |   FUR01_Z2_TEMP_L |   FUR01_Z2_TEMP_R |   FUR01_Z3_TEMP_L |   FUR01_Z3_TEMP_R |   FUR01_MAT_TEMP |   FUR01_EXH_TEMP |   FUR01_LINE_SPEED |
|:-----------|------------------:|------------------:|------------------:|------------------:|------------------:|------------------:|-----------------:|-----------------:|-------------------:|
| 2026-01-01 |              1195 |              1200 |              1225 |              1228 |              1248 |              1251 |             1182 |              640 |                 12 |
| 2026-01-02 |              1196 |              1201 |              1226 |              1229 |              1249 |              1252 |             1183 |              641 |                 12 |
| 2026-01-03 |              1194 |              1200 |              1224 |              1228 |              1247 |              1251 |             1181 |              640 |                 12 |
| 2026-01-04 |              11

### Step 1. 컬럼 분류

FUR로 시작하는 태그를 측정 대상별로 나눕니다.


In [29]:
fur_tags = tags[tags["tag"].str.startswith("FUR")]
step1_tags = fur_tags[["tag", "physical_qty", "unit", "circuit_position", "sampling_sec"]]

print(step1_tags.to_markdown(index=False))


| tag              | physical_qty   | unit   | circuit_position           |   sampling_sec |
|:-----------------|:---------------|:-------|:---------------------------|---------------:|
| FUR01_Z1_TEMP_L  | 온도           | degC   | 1존 좌측 열전대            |            300 |
| FUR01_Z1_TEMP_R  | 온도           | degC   | 1존 우측 열전대            |            300 |
| FUR01_Z2_TEMP_L  | 온도           | degC   | 2존 좌측 열전대            |            300 |
| FUR01_Z2_TEMP_R  | 온도           | degC   | 2존 우측 열전대            |            300 |
| FUR01_Z3_TEMP_L  | 온도           | degC   | 3존 좌측 열전대            |            300 |
| FUR01_Z3_TEMP_R  | 온도           | degC   | 3존 우측 열전대            |            300 |
| FUR01_EXH_TEMP   | 온도           | degC   | 배기 덕트 열전대           |            300 |
| FUR01_MAT_TEMP   | 온도           | degC   | 출구 소재 표면 비접촉 센서 |             60 |
| FUR01_LINE_SPEED | 속도           | m/min  | 라인 구동부                |             60 |


In [30]:
rows = []

atmosphere_tags = "FUR01_Z1_TEMP_L, FUR01_Z1_TEMP_R, FUR01_Z2_TEMP_L, FUR01_Z2_TEMP_R, FUR01_Z3_TEMP_L, FUR01_Z3_TEMP_R"
rows.append(["분위기 온도", atmosphere_tags])
rows.append(["소재 표면 온도", "FUR01_MAT_TEMP"])
rows.append(["배기가스 온도", "FUR01_EXH_TEMP"])
rows.append(["라인 속도", "FUR01_LINE_SPEED"])

step1_table = pd.DataFrame(rows, columns=["측정 대상", "해당 태그명"])
print(step1_table.to_markdown(index=False))


| 측정 대상      | 해당 태그명                                                                                          |
|:---------------|:-----------------------------------------------------------------------------------------------------|
| 분위기 온도    | FUR01_Z1_TEMP_L, FUR01_Z1_TEMP_R, FUR01_Z2_TEMP_L, FUR01_Z2_TEMP_R, FUR01_Z3_TEMP_L, FUR01_Z3_TEMP_R |
| 소재 표면 온도 | FUR01_MAT_TEMP                                                                                       |
| 배기가스 온도  | FUR01_EXH_TEMP                                                                                       |
| 라인 속도      | FUR01_LINE_SPEED                                                                                     |


### Step 2. 존별로 '앞 20일 구간'의 평균 편차(정상구간) 확인

존
- Z1, Z2, Z3

계산식  
좌우 온도 편차 = 우측 온도 - 좌측 온도


In [31]:
df["Z1_DIFF"] = df["FUR01_Z1_TEMP_R"] - df["FUR01_Z1_TEMP_L"]
df["Z2_DIFF"] = df["FUR01_Z2_TEMP_R"] - df["FUR01_Z2_TEMP_L"]
df["Z3_DIFF"] = df["FUR01_Z3_TEMP_R"] - df["FUR01_Z3_TEMP_L"]

normal_20 = df.head(20)

rows = []
rows.append(["Z1", normal_20["Z1_DIFF"].mean()])
rows.append(["Z2", normal_20["Z2_DIFF"].mean()])
rows.append(["Z3", normal_20["Z3_DIFF"].mean()])

normal_diff = pd.DataFrame(rows, columns=["존", "평균 편차"])
print(normal_diff.to_markdown(index=False))


| 존   |   평균 편차 |
|:-----|------------:|
| Z1   |         5.1 |
| Z2   |         3.1 |
| Z3   |         3.1 |


### Step 3. 시간이 지나며 편차가 어떻게 변하는지 확인

존(Z1, Z2, Z3)별로 아래 시점의 좌우 온도 편차를 비교하세요.

- 1일차
- 20일차
- 40일차
- 60일차


In [32]:
rows = []

row = df.iloc[0]
rows.append(["1일차", row["Z1_DIFF"], row["Z2_DIFF"], row["Z3_DIFF"]])

row = df.iloc[19]
rows.append(["20일차", row["Z1_DIFF"], row["Z2_DIFF"], row["Z3_DIFF"]])

row = df.iloc[39]
rows.append(["40일차", row["Z1_DIFF"], row["Z2_DIFF"], row["Z3_DIFF"]])

row = df.iloc[59]
rows.append(["60일차", row["Z1_DIFF"], row["Z2_DIFF"], row["Z3_DIFF"]])

diff_by_day = pd.DataFrame(rows, columns=["시점", "Z1 편차", "Z2 편차", "Z3 편차"])
print(diff_by_day.to_markdown(index=False))


| 시점   |   Z1 편차 |   Z2 편차 |   Z3 편차 |
|:-------|----------:|----------:|----------:|
| 1일차  |         5 |         3 |         3 |
| 20일차 |         6 |         4 |         4 |
| 40일차 |         6 |        14 |         4 |
| 60일차 |         6 |        24 |         4 |


In [33]:
z1_increase = df["Z1_DIFF"].iloc[59] - df["Z1_DIFF"].iloc[0]
z2_increase = df["Z2_DIFF"].iloc[59] - df["Z2_DIFF"].iloc[0]
z3_increase = df["Z3_DIFF"].iloc[59] - df["Z3_DIFF"].iloc[0]

rows = []
rows.append(["Z1", z1_increase])
rows.append(["Z2", z2_increase])
rows.append(["Z3", z3_increase])

increase_table = pd.DataFrame(rows, columns=["존", "편차 증가폭"])
print(increase_table.to_markdown(index=False))

max_increase_zone = "Z2"
print("가장 크게 편차가 증가한 존:", max_increase_zone)


| 존   |   편차 증가폭 |
|:-----|--------------:|
| Z1   |             1 |
| Z2   |            21 |
| Z3   |             1 |
가장 크게 편차가 증가한 존: Z2


### Step 4. 이상 위치 확인

편차가 가장 크게 증가한 존의 좌측 온도와 우측 온도를 비교하세요.


In [34]:
rows = []

row = df.iloc[0]
rows.append(["1일차", row["FUR01_Z2_TEMP_L"], row["FUR01_Z2_TEMP_R"]])

row = df.iloc[19]
rows.append(["20일차", row["FUR01_Z2_TEMP_L"], row["FUR01_Z2_TEMP_R"]])

row = df.iloc[39]
rows.append(["40일차", row["FUR01_Z2_TEMP_L"], row["FUR01_Z2_TEMP_R"]])

row = df.iloc[59]
rows.append(["60일차", row["FUR01_Z2_TEMP_L"], row["FUR01_Z2_TEMP_R"]])

side_compare = pd.DataFrame(rows, columns=["시점", "좌측 온도", "우측 온도"])
print(side_compare.to_markdown(index=False))


| 시점   |   좌측 온도 |   우측 온도 |
|:-------|------------:|------------:|
| 1일차  |        1225 |        1228 |
| 20일차 |        1224 |        1228 |
| 40일차 |        1214 |        1228 |
| 60일차 |        1204 |        1228 |


### Step 5. 소재 온도와 라인 속도 비교

아래 시점에서 소재 온도와 라인 속도를 확인하세요.

- 1일차
- 20일차
- 44일차
- 45일차
- 60일차


In [35]:
rows = []

row = df.iloc[0]
rows.append(["1일차", row["FUR01_MAT_TEMP"], row["FUR01_LINE_SPEED"]])

row = df.iloc[19]
rows.append(["20일차", row["FUR01_MAT_TEMP"], row["FUR01_LINE_SPEED"]])

row = df.iloc[43]
rows.append(["44일차", row["FUR01_MAT_TEMP"], row["FUR01_LINE_SPEED"]])

row = df.iloc[44]
rows.append(["45일차", row["FUR01_MAT_TEMP"], row["FUR01_LINE_SPEED"]])

row = df.iloc[59]
rows.append(["60일차", row["FUR01_MAT_TEMP"], row["FUR01_LINE_SPEED"]])

mat_speed_compare = pd.DataFrame(rows, columns=["시점", "소재 온도", "라인 속도"])
print(mat_speed_compare.to_markdown(index=False))


| 시점   |   소재 온도 |   라인 속도 |
|:-------|------------:|------------:|
| 1일차  |        1182 |          12 |
| 20일차 |        1181 |          12 |
| 44일차 |        1180 |          12 |
| 45일차 |        1172 |          14 |
| 60일차 |        1169 |          14 |



질문  
45일차 전후로 라인 속도와 소재 온도는 어떻게 변했나요?

→ 45일차부터 라인 속도가 12에서 14로 증가했고, 소재 온도는 더 크게 떨어졌다.

### Step 6. 최종 해석

1. Z2의 좌우 온도 편차가 커진 원인을 어떻게 해석할 수 있을까요?

→ 좌측 온도가 계속 내려갔기 때문이다.

2. 소재 온도가 떨어진 원인은 가열로 온도 저하일까요, 아니면 라인 속도 변화일까요?

→ 라인 속도가 증가하면서 소재가 가열로에 머무는 시간이 줄어든 것이 원인이 될 수 있다. 또한 Z2 좌측 온도가 떨어진 것도 같이 영향을 준 것으로 보인다.

3. 전체 결과를 2~3문장으로 정리하세요.

→ Z2는 우측 온도보다 좌측 온도가 계속 내려가면서 좌우 편차가 가장 크게 커졌다.  
→ 45일차 이후 라인 속도가 증가했고 소재 온도도 더 낮아졌다. 그래서 소재 온도 하락은 라인 속도 증가와 Z2 좌측 온도 저하가 같이 영향을 준 것으로 볼 수 있다.
